In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


In [4]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import HumanMessage,SystemMessage,AIMessage
from langgraph.checkpoint.memory import InMemorySaver

model = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)

agent = create_agent(
    model=model,
    tools = [],
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [5]:
config = {"configurable":{"thread":"test-1"}}

In [6]:
#Alternative test data
questions= [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"message{response}")
    print(f"messages{len(response['messages'])}")

message{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='9f86756c-6af8-4579-93dd-89e324c44fb1'), AIMessage(content='2 + 2 = 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.008751005, 'completion_tokens_details': None, 'prompt_time': 0.008323205, 'prompt_tokens_details': None, 'queue_time': 0.162507126, 'total_time': 0.01707421}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eab1a-6af1-7fa0-9b18-19de8e8811ed-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51})]}
messages2
message{'messages': [HumanMessage(content='What is 10*5?', additional_kwargs={}, response_metadata={}, id='f3ca1a60-fba1-4aa2-929a-4273c8dd0c51'), AIMessage(cont

In [11]:
##token size

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage,SystemMessage
from langchain_core.tools import tool
 
@tool
def search_hostels(city:str)->str:
    '''Serach hostels-retuerns long response to use tokens'''
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

 

llm = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider='groq'
)

agent = create_agent(
    model=llm,
    tools=[search_hostels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model= llm,
            trigger=("tokens",550),
            keep=("tokens",200)
        ),
    ]
    
    
)

config = {"configurable":{"thread_id":"test-1"}}

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4

In [ ]:
cities = ['Paris','Dubai','New york']
for city in cities:
    response = agent.invoke({"messages":[HumanMessage(content=f"find hostels in the {city}")]},
                            config=config)
    tokens = count_tokens(response["messages"])
    print(f"{city}:~{tokens}tokens,{len(response['messages'])}messages")
    print(f"{response["messages"]}")